# Task 5: Quantization & Edge Deployment (Workbench)

Merge LoRA → Quantize to INT4 → Quality/Speed comparison → Export for edge.

## Setup

In [ ]:
import os, sys, json, re, time, torch, gc
import numpy as np
from PIL import Image
from collections import defaultdict, Counter
import matplotlib.pyplot as plt

# Add task4 config to path
sys.path.insert(0, '../../task4-fine-tuning/granulometry')
from config import *

LORA_PATH = '../../task4-fine-tuning/granulometry/lora_augmented'
MERGED_PATH = './merged_model'
QUANTIZED_PATH = './quantized_model_int4'

for i in range(torch.cuda.device_count()):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB')


## Step 1: Merge LoRA into Base Model

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import PeftModel

print('Loading base model...')
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, device_map='auto', torch_dtype=torch.bfloat16)

print(f'Loading LoRA adapter from {LORA_PATH}...')
model = PeftModel.from_pretrained(base_model, LORA_PATH)

print('Merging LoRA into base model...')
model = model.merge_and_unload()

print(f'Saving merged model to {MERGED_PATH}...')
model.save_pretrained(MERGED_PATH)

# Also save processor
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256*28*28, max_pixels=512*28*28)
processor.save_pretrained(MERGED_PATH)

# Check size
import shutil
merged_size = sum(f.stat().st_size for f in os.scandir(MERGED_PATH) if f.is_file()) / 1e9
print(f'\nMerged model size: {merged_size:.2f} GB')
print('Merge complete. This is a standalone model — no adapter needed.')

del model, base_model; gc.collect(); torch.cuda.empty_cache()


## Step 2: Quantize to INT4
Using AutoAWQ or AutoGPTQ for 4-bit quantization.

In [ ]:
# Option A: AWQ quantization (recommended for inference speed)
# pip install autoawq
try:
    from awq import AutoAWQForCausalLM
    print('Using AWQ quantization')
    
    quant_config = {
        'zero_point': True,
        'q_group_size': 128,
        'w_bit': 4,
        'version': 'GEMM',
    }
    
    print(f'Loading merged model from {MERGED_PATH}...')
    model = AutoAWQForCausalLM.from_pretrained(MERGED_PATH)
    tokenizer = AutoProcessor.from_pretrained(MERGED_PATH)
    
    print('Quantizing (this may take 10-30 minutes)...')
    model.quantize(tokenizer, quant_config=quant_config)
    
    print(f'Saving quantized model to {QUANTIZED_PATH}...')
    model.save_quantized(QUANTIZED_PATH)
    tokenizer.save_pretrained(QUANTIZED_PATH)
    
    quant_size = sum(f.stat().st_size for f in os.scandir(QUANTIZED_PATH) if f.is_file()) / 1e9
    print(f'Quantized model size: {quant_size:.2f} GB')
    print(f'Compression ratio: {merged_size/quant_size:.1f}x')
    
    del model; gc.collect(); torch.cuda.empty_cache()
    USE_AWQ = True

except ImportError:
    print('AWQ not available. Trying GPTQ...')
    USE_AWQ = False


In [ ]:
# Option B: GPTQ quantization (fallback)
if not USE_AWQ:
    from transformers import GPTQConfig
    
    print(f'Loading merged model for GPTQ quantization...')
    gptq_config = GPTQConfig(bits=4, dataset='c4', tokenizer=None)
    
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MERGED_PATH, quantization_config=gptq_config, device_map='auto')
    
    print(f'Saving quantized model to {QUANTIZED_PATH}...')
    model.save_pretrained(QUANTIZED_PATH)
    processor.save_pretrained(QUANTIZED_PATH)
    
    quant_size = sum(f.stat().st_size for f in os.scandir(QUANTIZED_PATH) if f.is_file()) / 1e9
    print(f'Quantized model size: {quant_size:.2f} GB')
    print(f'Compression ratio: {merged_size/quant_size:.1f}x')
    
    del model; gc.collect(); torch.cuda.empty_cache()


## Step 3: Quality Check
Compare BF16 merged vs INT4 quantized on the same 108 test images.

In [ ]:
def parse_response(raw):
    if not raw: return None
    raw = raw.replace('<','').replace('>','')
    raw = re.sub(r'```json\s*','',raw); raw = re.sub(r'```\s*','',raw).strip()
    try:
        obj = json.loads(raw)
        if isinstance(obj, dict): return obj
    except: pass
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    if m:
        try: return json.loads(m.group())
        except: pass
    sm = re.search(r'"max_particle_size_mm"\s*:\s*(\d+)', raw)
    gm = re.search(r'"grading"\s*:\s*"(\w+)"', raw)
    if sm and gm: return {'max_particle_size_mm': int(sm.group(1)), 'grading': gm.group(1)}
    return None

def run_eval(model, processor, manifest, label='', max_dim=MAX_DIM):
    model.eval()
    results=[]; cs=0; cg=0; vj=0; tt=0
    peak_mem = 0
    for i, entry in enumerate(manifest):
        img_path = os.path.join(TEST_DIR, entry['image'])
        if not os.path.exists(img_path): continue
        image = Image.open(img_path).convert('RGB')
        scale = min(max_dim/max(image.size), 1.0)
        gsd = ORIGINAL_GSD * scale
        prompt = make_prompt(gsd)
        msgs = [{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':prompt}]}]
        text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[image], return_tensors='pt', padding=True).to(model.device)
        torch.cuda.reset_peak_memory_stats()
        t=time.time()
        with torch.no_grad(): ids = model.generate(**inputs, max_new_tokens=128, temperature=0.1, do_sample=True)
        elapsed=time.time()-t; tt+=elapsed
        peak_mem = max(peak_mem, torch.cuda.max_memory_allocated()/1e9)
        out = processor.batch_decode(ids[:,inputs.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()
        del inputs, ids; image.close(); torch.cuda.empty_cache()
        parsed = parse_response(out)
        gs=entry['max_particle_size_mm']; gg=entry['grading']; so=False; go=False
        if parsed:
            vj+=1; ps=parsed.get('max_particle_size_mm')
            if isinstance(ps,str): ps=int(ps) if ps.isdigit() else None
            if ps==gs: so=True; cs+=1
            if parsed.get('grading','').lower()==gg: go=True; cg+=1
        results.append({'image':entry['image'],'class':entry['class'],'gt_size':gs,'gt_grading':gg,
            'predicted':parsed,'raw':out,'size_correct':so,'grading_correct':go,'valid_json':parsed is not None,'time_s':round(elapsed,2)})
        if (i+1)%20==0:
            n=i+1; print(f'  [{n}/{len(manifest)}] Size:{cs}/{n}({cs/n*100:.0f}%) Grade:{cg}/{n}({cg/n*100:.0f}%)')
    return results, cs, cg, vj, tt, peak_mem

with open(TEST_MANIFEST) as f: test_manifest = json.load(f)
quick = [next(e for e in test_manifest if e['class']==cls) for cls in sorted(GT.keys())]
print(f'Test: {len(test_manifest)} images, Quick: {len(quick)} images')


### BF16 Merged Model

In [ ]:
print('Loading BF16 merged model...')
processor_bf16 = AutoProcessor.from_pretrained(MERGED_PATH, min_pixels=256*28*28, max_pixels=512*28*28)
model_bf16 = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MERGED_PATH, device_map='auto', torch_dtype=torch.bfloat16)

# Quick eval
print('\nQuick eval (9 images):')
r9, cs9, cg9, vj9, tt9, mem9 = run_eval(model_bf16, processor_bf16, quick, 'BF16')
print(f'Size: {cs9}/{len(r9)} | Grading: {cg9}/{len(r9)} | Peak VRAM: {mem9:.1f} GB')
for x in r9:
    p=x['predicted']; ps=p.get('max_particle_size_mm','?') if p else '?'; pg=p.get('grading','?') if p else '?'
    sv='✓' if x['size_correct'] else '✗'; gv='✓' if x['grading_correct'] else '✗'
    print(f'  {x["class"]:>3} GT:{x["gt_size"]}mm/{x["gt_grading"]} Pred:{ps}mm/{pg} {sv}{gv}')

# Full eval
print('\nFull eval (108 images):')
r_bf16, cs_bf16, cg_bf16, vj_bf16, tt_bf16, mem_bf16 = run_eval(model_bf16, processor_bf16, test_manifest, 'BF16')
n=len(r_bf16); both_bf16=sum(1 for x in r_bf16 if x['size_correct'] and x['grading_correct'])
print(f'\nBF16: Size={cs_bf16}/{n}({cs_bf16/n*100:.1f}%) Grade={cg_bf16}/{n}({cg_bf16/n*100:.1f}%) Both={both_bf16}/{n}({both_bf16/n*100:.1f}%)')
print(f'Avg time: {tt_bf16/n:.2f}s | Peak VRAM: {mem_bf16:.1f} GB')

del model_bf16; gc.collect(); torch.cuda.empty_cache()


### INT4 Quantized Model

In [ ]:
print('Loading INT4 quantized model...')
# Load method depends on quantization format
try:
    # AWQ
    from awq import AutoAWQForCausalLM
    model_int4 = AutoAWQForCausalLM.from_quantized(QUANTIZED_PATH, fuse_layers=True)
    print('Loaded AWQ quantized model')
except:
    # GPTQ or standard
    model_int4 = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        QUANTIZED_PATH, device_map='auto', torch_dtype=torch.float16)
    print('Loaded quantized model (standard)')

processor_int4 = AutoProcessor.from_pretrained(QUANTIZED_PATH, min_pixels=256*28*28, max_pixels=512*28*28)

# Quick eval
print('\nQuick eval (9 images):')
r9q, cs9q, cg9q, vj9q, tt9q, mem9q = run_eval(model_int4, processor_int4, quick, 'INT4')
print(f'Size: {cs9q}/{len(r9q)} | Grading: {cg9q}/{len(r9q)} | Peak VRAM: {mem9q:.1f} GB')
for x in r9q:
    p=x['predicted']; ps=p.get('max_particle_size_mm','?') if p else '?'; pg=p.get('grading','?') if p else '?'
    sv='✓' if x['size_correct'] else '✗'; gv='✓' if x['grading_correct'] else '✗'
    print(f'  {x["class"]:>3} GT:{x["gt_size"]}mm/{x["gt_grading"]} Pred:{ps}mm/{pg} {sv}{gv}')

# Full eval
print('\nFull eval (108 images):')
r_int4, cs_int4, cg_int4, vj_int4, tt_int4, mem_int4 = run_eval(model_int4, processor_int4, test_manifest, 'INT4')
n=len(r_int4); both_int4=sum(1 for x in r_int4 if x['size_correct'] and x['grading_correct'])
print(f'\nINT4: Size={cs_int4}/{n}({cs_int4/n*100:.1f}%) Grade={cg_int4}/{n}({cg_int4/n*100:.1f}%) Both={both_int4}/{n}({both_int4/n*100:.1f}%)')
print(f'Avg time: {tt_int4/n:.2f}s | Peak VRAM: {mem_int4:.1f} GB')

del model_int4; gc.collect(); torch.cuda.empty_cache()


## Step 4: BF16 vs INT4 Comparison

In [ ]:
n = len(r_bf16)
both_bf16 = sum(1 for x in r_bf16 if x['size_correct'] and x['grading_correct'])
both_int4 = sum(1 for x in r_int4 if x['size_correct'] and x['grading_correct'])

print(f'{"Metric":<20} {"BF16":>10} {"INT4":>10} {"Delta":>10}')
print('=' * 52)
rows = [
    ('Model size', f'{merged_size:.2f} GB', f'{quant_size:.2f} GB', f'{merged_size/quant_size:.1f}x smaller'),
    ('Peak VRAM', f'{mem_bf16:.1f} GB', f'{mem_int4:.1f} GB', f'{mem_bf16-mem_int4:.1f} GB saved'),
    ('Avg time/image', f'{tt_bf16/n:.2f}s', f'{tt_int4/n:.2f}s', f'{(tt_bf16/n)/(tt_int4/n):.1f}x faster' if tt_int4>0 else '—'),
    ('Size accuracy', f'{cs_bf16/n*100:.1f}%', f'{cs_int4/n*100:.1f}%', f'{(cs_int4-cs_bf16)/n*100:+.1f}%'),
    ('Grading accuracy', f'{cg_bf16/n*100:.1f}%', f'{cg_int4/n*100:.1f}%', f'{(cg_int4-cg_bf16)/n*100:+.1f}%'),
    ('Both correct', f'{both_bf16/n*100:.1f}%', f'{both_int4/n*100:.1f}%', f'{(both_int4-both_bf16)/n*100:+.1f}%'),
    ('JSON validity', f'{vj_bf16/n*100:.0f}%', f'{vj_int4/n*100:.0f}%', '—'),
    ('FPS', f'{n/tt_bf16:.2f}', f'{n/tt_int4:.2f}', f'{(n/tt_int4)/(n/tt_bf16):.1f}x'),
]
for label, bf, it, delta in rows:
    print(f'{label:<20} {bf:>10} {it:>10} {delta:>10}')

# Quality degradation check
size_drop = (cs_bf16 - cs_int4) / n * 100
grade_drop = (cg_bf16 - cg_int4) / n * 100
print(f'\nQuality degradation: size {size_drop:+.1f}%, grading {grade_drop:+.1f}%')
print(f'Target: <5% degradation → {"PASS" if abs(size_drop)<5 and abs(grade_drop)<5 else "FAIL"}')


## Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Accuracy comparison
metrics = ['Size', 'Grading', 'Both']
bf_vals = [cs_bf16/n*100, cg_bf16/n*100, both_bf16/n*100]
it_vals = [cs_int4/n*100, cg_int4/n*100, both_int4/n*100]
x = np.arange(len(metrics)); w = 0.35
axes[0].bar(x-w/2, bf_vals, w, label='BF16', color='steelblue')
axes[0].bar(x+w/2, it_vals, w, label='INT4', color='coral')
axes[0].set_xticks(x); axes[0].set_xticklabels(metrics)
axes[0].set_ylabel('%'); axes[0].set_title('Accuracy: BF16 vs INT4'); axes[0].legend(); axes[0].set_ylim(0,110)

# Speed comparison
axes[1].bar(['BF16', 'INT4'], [tt_bf16/n, tt_int4/n], color=['steelblue','coral'])
axes[1].set_ylabel('Seconds/image'); axes[1].set_title('Inference Speed')

# Memory comparison
axes[2].bar(['BF16\n(model)', 'INT4\n(model)', 'BF16\n(peak VRAM)', 'INT4\n(peak VRAM)'],
    [merged_size, quant_size, mem_bf16, mem_int4], color=['steelblue','coral','steelblue','coral'], alpha=[1,1,0.5,0.5])
axes[2].set_ylabel('GB'); axes[2].set_title('Size & Memory')

plt.suptitle('Task 5: BF16 vs INT4 Quantized', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()


## Save Results

In [ ]:
for label, r, cs, cg, vj, tt, mem in [
    ('bf16', r_bf16, cs_bf16, cg_bf16, vj_bf16, tt_bf16, mem_bf16),
    ('int4', r_int4, cs_int4, cg_int4, vj_int4, tt_int4, mem_int4),
]:
    n=len(r); both=sum(1 for x in r if x['size_correct'] and x['grading_correct'])
    with open(f'results_{label}.json','w') as f:
        json.dump({'model':MODEL_ID,'precision':label,'total_images':n,
            'json_validity_pct':round(vj/n*100,1),'size_accuracy_pct':round(cs/n*100,1),
            'grading_accuracy_pct':round(cg/n*100,1),'both_correct_pct':round(both/n*100,1),
            'avg_inference_time_s':round(tt/n,2),'peak_vram_gb':round(mem,1),
            'results':r}, f, indent=2)
    print(f'Saved results_{label}.json')

print(f'\nQuantized model ready at {QUANTIZED_PATH}/')
print(f'Copy to edge VM for deployment testing.')
